<a href="https://colab.research.google.com/github/nejumi/wandb-physicalai-workshop/blob/main/notebooks/02_isaac_lab_wandb_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Isaac Lab × W&B — Cartpole PPO quickstart

Isaac Lab の standalone Newton backend で `Isaac-Cartpole` を短く学習します。GUI 版 Isaac Sim は使いません。

完了時には同じ W&B Run に次の3種類が揃います。

- Metrics: reward、episode length、PPO loss
- Model Artifact: `isaaclab-cartpole-policy`
- Video: `eval/simulation`

Colab の **ランタイム > ランタイムのタイプを変更 > T4 GPU** を選び、上から順に実行してください。初回セットアップは5〜10分、学習は2〜3分が目安です。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
import sys
print('Python:', sys.version)

## 1. 教材と Isaac Lab を準備する

再現性のため Isaac Lab は講義前に確認した commit に固定します。大量の依存を取得するため、このセルが最も時間を使います。

In [ ]:
from pathlib import Path
import os

workshop = Path('/content/wandb-physicalai-workshop')
isaaclab = Path('/content/IsaacLab')
if not workshop.exists():
    !git clone -q https://github.com/nejumi/wandb-physicalai-workshop.git {workshop}
if not isaaclab.exists():
    !git clone -q --no-checkout https://github.com/isaac-sim/IsaacLab.git {isaaclab}
    !git -C {isaaclab} fetch -q --depth 1 origin 7bca8097a5ed27c26744686c84355d1786769e81
    !git -C {isaaclab} checkout -q 7bca8097a5ed27c26744686c84355d1786769e81
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!cd {isaaclab} && uv sync --locked --extra wandb --extra video

## 2. W&B の接続情報を設定する

Colab Secrets の `WANDB_API_KEY` を優先し、なければ入力します。Dedicated Cloud の場合は `WANDB_BASE_URL` も登録してください。

In [ ]:
import getpass, os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

os.environ['WANDB_API_KEY'] = secret('WANDB_API_KEY') or getpass.getpass('WANDB_API_KEY: ')
os.environ['WANDB_BASE_URL'] = (secret('WANDB_BASE_URL') or 'https://api.wandb.ai').rstrip('/')
os.environ['WANDB_PROJECT'] = secret('WANDB_PROJECT') or 'physical-ai-workshop'
entity = secret('WANDB_ENTITY')
if entity:
    os.environ['WANDB_ENTITY'] = entity
print('project:', os.environ['WANDB_PROJECT'])

## 3. Cartpole PPO を2 iterationsだけ実行する

初回は CUDA kernel compile に時間がかかります。Isaac Lab native logger が学習曲線を記録し、学習後に同じ Run を resume して最新 checkpoint と mp4 を追加します。

In [ ]:
os.environ['ISAACLAB_DIR'] = str(isaaclab)
os.environ['ISAACLAB_ITERATIONS'] = '2'
os.environ['ISAACLAB_NUM_ENVS'] = '32'
!PATH=/root/.local/bin:$PATH bash {workshop}/scripts/10_run_isaac_colab.sh

## 4. simulation video を確認する

Notebook で動画を確認した後、出力された W&B Run URL を開きます。Charts、Artifacts、Media が同じ Run にあることを確認してください。

In [ ]:
from IPython.display import Video, display
videos = sorted(isaaclab.glob('logs/rsl_rl/wandb-course-cartpole/*/videos/train/*.mp4'))
print(videos[-1] if videos else 'video not found')
display(Video(str(videos[-1]), embed=True)) if videos else None